In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Drug–Target Interaction Prediction using Graph Neural Networks and Protein Language Models

## Motivation

Drug discovery is one of the most expensive and time-consuming processes in biomedical research. Experimental screening of every possible drug against every possible protein is impractical.

Drug–Target Interaction (DTI) prediction aims to computationally estimate whether a drug molecule interacts with a target protein. Accurate prediction of these interactions accelerates drug discovery, drug repurposing, and personalized medicine.

---

## Objective

Develop a deep learning framework that predicts the interaction between drugs and proteins by combining:

- Graph Neural Networks (GNNs) for molecular graph representation.
- Protein Language Models (ESM2) for protein sequence representation.
- A multimodal neural network for interaction prediction.

---

## Learning Objectives

Throughout this project, we will learn:

- Molecular graph construction
- Graph Neural Networks
- Protein Language Models
- Representation Learning
- Multimodal Deep Learning
- Drug Discovery
- Explainable AI

---

## Dataset

We use the **Davis Drug–Target Interaction Dataset**, a benchmark dataset widely used for evaluating DTI prediction models.

---

## Workflow

Drug (SMILES)
        │
        ▼
RDKit
        │
        ▼
Molecular Graph
        │
        ▼
Graph Neural Network
        │
        ▼
Drug Embedding
                    │
                    ▼
             Feature Fusion
                    ▲
                    │
Protein Sequence    │
        │           │
      ESM2          │
        │           │
Protein Embedding   │
                    ▼
          Interaction Prediction

## Environment Setup

This notebook is designed for execution on Kaggle using a Tesla T4 GPU.

Main libraries:

- PyTorch
- PyTorch Geometric
- RDKit
- Transformers
- Pandas
- NumPy
- Scikit-Learn

In [ ]:
##############################################################
# 2.1 Install Dependencies
##############################################################

!pip -q install torch-geometric
!pip -q install rdkit
!pip -q install transformers
!pip -q install accelerate 

In [ ]:
##############################################################
# Install Biopython
##############################################################

!pip install -q biopython

In [ ]:
##############################################################
# 2.2 Imports
##############################################################

import os
import gc
import random
import warnings

from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import torch

warnings.filterwarnings("ignore")

In [ ]:
##############################################################
# Biopython Imports
##############################################################

from Bio.SeqUtils.ProtParam import ProteinAnalysis
from Bio.SeqUtils import molecular_weight

print("Biopython installed successfully!")

In [ ]:
from pathlib import Path

checkpoint_dir = Path(CHECKPOINT_DIR)

deleted = 0

for file in checkpoint_dir.glob("*.pt"):
    file.unlink()
    print(f"Deleted: {file.name}")
    deleted += 1

print(f"\nTotal checkpoint files deleted: {deleted}")

In [ ]:
##############################################################
# 2.3 Reproducibility
##############################################################

SEED = 42

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

torch.cuda.manual_seed_all(SEED)

In [ ]:
##############################################################
# 2.4 Device Information
##############################################################

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("="*60)

print("Device :", device)

if torch.cuda.is_available():

    print("GPU :", torch.cuda.get_device_name(0))

    print(
        f"GPU Memory : "
        f"{torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB"
    )

print("="*60)

In [ ]:
##############################################################
# 2.5 Directory Structure
##############################################################

PROJECT_ROOT = Path("DTI_Project")

RAW_DIR = PROJECT_ROOT / "raw"

PROCESSED_DIR = PROJECT_ROOT / "processed"

ARTIFACT_DIR = PROJECT_ROOT / "artifacts"

MODEL_DIR = PROJECT_ROOT / "models"

FIGURE_DIR = PROJECT_ROOT / "figures"

for directory in [

    PROJECT_ROOT,

    RAW_DIR,

    PROCESSED_DIR,

    ARTIFACT_DIR,

    MODEL_DIR,

    FIGURE_DIR

]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )

print("Project folders created successfully.")

# Section 3: Loading the Davis Dataset

In this section we load the raw Davis dataset and inspect its contents before performing any preprocessing.

The dataset consists of:

- Drug molecules represented as SMILES strings.
- Protein sequences.
- Drug–protein binding affinity matrix.
- Official train/test folds used in the DeepDTA benchmark.

At this stage, no preprocessing is performed. We only verify that the dataset has been loaded correctly.

In [ ]:
##############################################################
# 3.1 Imports
##############################################################

import json
import pickle

from pathlib import Path

import numpy as np
import pandas as pd

In [ ]:
##############################################################
# 3.2 Dataset Path
##############################################################

DATASET_ROOT = Path("/kaggle/working/DeepDTA/data/davis")

print(DATASET_ROOT)

In [ ]:
##############################################################
# Download Davis Dataset
##############################################################

!git clone https://github.com/hkmztrk/DeepDTA.git

In [ ]:
##############################################################
# 3.3 Load Drug Molecules
##############################################################

drug_file = DATASET_ROOT / "ligands_can.txt"

with open(drug_file, "r") as f:
    drugs = json.load(f)

drug_df = pd.DataFrame({
    "Drug_ID": list(drugs.keys()),
    "SMILES": list(drugs.values())
})

print("=" * 60)
print("Drug Dataset")
print("=" * 60)

print(f"Number of Drugs : {len(drug_df)}")

drug_df.head()

In [ ]:
##############################################################
# 3.4 Load Protein Sequences
##############################################################

protein_file = DATASET_ROOT / "proteins.txt"

with open(protein_file, "r") as f:
    proteins = json.load(f)

protein_df = pd.DataFrame({
    "Protein_ID": list(proteins.keys()),
    "Sequence": list(proteins.values())
})

print("=" * 60)
print("Protein Dataset")
print("=" * 60)

print(f"Number of Proteins : {len(protein_df)}")

protein_df.head()

In [ ]:
##############################################################
# 3.5 Load Binding Affinity Matrix
##############################################################

import pickle
import numpy as np

affinity_file = DATASET_ROOT / "Y"

with open(affinity_file, "rb") as f:
    affinity_matrix = pickle.load(
        f,
        encoding="latin1"
    )

affinity_matrix = np.array(affinity_matrix)

print("=" * 60)
print("Binding Affinity Matrix")
print("=" * 60)

print("Shape :", affinity_matrix.shape)
print("Data Type :", affinity_matrix.dtype)

In [ ]:
##############################################################
# 3.8 Load Official Train/Test Splits
##############################################################

fold_path = DATASET_ROOT / "folds"

with open(fold_path / "train_fold_setting1.txt") as f:
    train_folds = json.load(f)

with open(fold_path / "test_fold_setting1.txt") as f:
    test_fold = json.load(f)

print("=" * 60)
print("Official Dataset Splits")
print("=" * 60)

print(f"Training Folds : {len(train_folds)}")
print(f"Test Samples   : {len(test_fold)}")

## 4.1 Dataset Dimensions

Before building a deep learning model, it is useful to understand the overall size of the dataset.

The Davis dataset consists of:

- Drug molecules
- Protein targets
- Drug–protein interaction matrix

This provides an overview of the prediction problem.

In [ ]:
##############################################################
# 4.1 Dataset Dimensions
##############################################################

print("=" * 60)
print("Dataset Dimensions")
print("=" * 60)

print(f"Number of Drugs      : {len(drug_df):,}")
print(f"Number of Proteins   : {len(protein_df):,}")
print(f"Affinity Matrix Size : {affinity_matrix.shape}")
print(f"Possible Interactions: {affinity_matrix.size:,}")

## 4.2 Drug Statistics

Each drug is represented using a SMILES string.

A SMILES string is a textual representation of a molecular structure.

We first inspect a few examples before converting them into molecular graphs.

In [ ]:
##############################################################
# 4.2 Drug Statistics
##############################################################

drug_df["SMILES_Length"] = drug_df["SMILES"].str.len()

display(drug_df.head())

print()

print(drug_df["SMILES_Length"].describe())

In [ ]:
##############################################################
# 4.3 SMILES Length Distribution
##############################################################

plt.figure(figsize=(8,5))

plt.hist(
    drug_df["SMILES_Length"],
    bins=20,
    edgecolor="black"
)

plt.xlabel("SMILES Length")

plt.ylabel("Count")

plt.title("Distribution of SMILES Length")

plt.show()

## 4.4 Protein Statistics

Each protein is represented by its amino acid sequence.

The sequence length varies considerably across proteins.

Understanding this distribution helps determine the maximum sequence length required for the protein encoder.

In [ ]:
##############################################################
# 4.4 Protein Statistics
##############################################################

protein_df["Sequence_Length"] = protein_df["Sequence"].str.len()

display(protein_df.head())

print()

print(protein_df["Sequence_Length"].describe())

In [ ]:
##############################################################
# 4.5 Protein Length Distribution
##############################################################

plt.figure(figsize=(8,5))

plt.hist(
    protein_df["Sequence_Length"],
    bins=30,
    edgecolor="black"
)

plt.xlabel("Protein Length")

plt.ylabel("Count")

plt.title("Protein Length Distribution")

plt.show()

In [ ]:
##############################################################
# 4.6 Affinity Statistics
##############################################################

##############################################################
# Affinity Statistics (All Interactions)
##############################################################

valid_affinity = affinity_matrix.flatten()

print("=" * 60)
print("Affinity Statistics")
print("=" * 60)
print(f"Total interactions : {len(valid_affinity):,}")
print(f"Minimum affinity   : {valid_affinity.min():.4f}")
print(f"Maximum affinity   : {valid_affinity.max():.4f}")
print(f"Mean affinity      : {valid_affinity.mean():.4f}")
print(f"Median affinity    : {np.median(valid_affinity):.4f}")

In [ ]:
##############################################################
# 4.7 Affinity Distribution
##############################################################

plt.figure(figsize=(8,5))

plt.hist(
    valid_affinity,
    bins=40,
    edgecolor="black"
)

plt.xlabel("Binding Affinity")

plt.ylabel("Count")

plt.title("Distribution of Binding Affinity")

plt.show()

In [ ]:
##############################################################
# 4.8 Missing Values
##############################################################

missing = np.isnan(affinity_matrix)

print("="*60)

print("Missing Values")

print("="*60)

print(f"Missing Entries : {missing.sum():,}")

print(f"Observed Entries: {(~missing).sum():,}")

In [ ]:
##############################################################
# 4.9 Duplicate Check
##############################################################

print("="*60)

print("Duplicate Check")

print("="*60)

print("Duplicate Drugs   :", drug_df.duplicated("SMILES").sum())

print("Duplicate Proteins:", protein_df.duplicated("Sequence").sum())

# Section 5: Drug Molecular Analysis

SMILES strings are compact textual representations of molecules.

Before converting them into graph structures for a Graph Neural Network, we first analyze their chemical properties.

Using RDKit, each SMILES string is converted into a molecular object from which various physicochemical descriptors are extracted.

These descriptors help us understand the diversity and complexity of the dataset and are commonly used in cheminformatics.

In [ ]:
import sys

print(sys.executable)

!pip show rdkit

In [ ]:
##############################################################
# 5.1 RDKit Imports
##############################################################

from rdkit import Chem

from rdkit.Chem import Descriptors

from rdkit.Chem import Lipinski

from rdkit.Chem import rdMolDescriptors

In [ ]:
##############################################################
# 5.2 Convert SMILES into RDKit Molecules
##############################################################

drug_df["Molecule"] = drug_df["SMILES"].apply(
    Chem.MolFromSmiles
)

invalid = drug_df["Molecule"].isnull().sum()

print("=" * 60)
print("SMILES Conversion")
print("=" * 60)

print(f"Total Drugs      : {len(drug_df)}")
print(f"Valid Molecules  : {len(drug_df)-invalid}")
print(f"Invalid Molecules: {invalid}")

In [ ]:
##############################################################
# 5.3 Extract Molecular Descriptors
##############################################################

def molecular_features(mol):

    return pd.Series({

        "NumAtoms": mol.GetNumAtoms(),

        "NumBonds": mol.GetNumBonds(),

        "MolWeight": Descriptors.MolWt(mol),

        "LogP": Descriptors.MolLogP(mol),

        "TPSA": rdMolDescriptors.CalcTPSA(mol),

        "RotatableBonds": Lipinski.NumRotatableBonds(mol),

        "HDonors": Lipinski.NumHDonors(mol),

        "HAcceptors": Lipinski.NumHAcceptors(mol),

        "Rings": rdMolDescriptors.CalcNumRings(mol)

    })

features = drug_df["Molecule"].apply(
    molecular_features
)

drug_df = pd.concat(
    [drug_df, features],
    axis=1
)

In [ ]:
##############################################################
# 5.4 Drug Feature Table
##############################################################

display(
    drug_df.head()
)

In [ ]:
##############################################################
# 5.5 Descriptor Statistics
##############################################################

columns = [

    "NumAtoms",

    "NumBonds",

    "MolWeight",

    "LogP",

    "TPSA",

    "RotatableBonds",

    "HDonors",

    "HAcceptors",

    "Rings"

]

drug_df[columns].describe()

In [ ]:
##############################################################
# 5.6 Molecular Weight
##############################################################

plt.figure(figsize=(8,5))

plt.hist(
    drug_df["MolWeight"],
    bins=15,
    edgecolor="black"
)

plt.xlabel("Molecular Weight")

plt.ylabel("Count")

plt.title("Distribution of Molecular Weight")

plt.show()

In [ ]:
##############################################################
# 5.7 LogP
##############################################################

plt.figure(figsize=(8,5))

plt.hist(
    drug_df["LogP"],
    bins=15,
    edgecolor="black"
)

plt.xlabel("LogP")

plt.ylabel("Count")

plt.title("Distribution of LogP")

plt.show()

In [ ]:
##############################################################
# 5.8 Number of Atoms
##############################################################

plt.figure(figsize=(8,5))

plt.hist(
    drug_df["NumAtoms"],
    bins=15,
    edgecolor="black"
)

plt.xlabel("Atoms")

plt.ylabel("Count")

plt.title("Distribution of Number of Atoms")

plt.show()

In [ ]:
##############################################################
# 5.9 Correlation Matrix
##############################################################

corr = drug_df[columns].corr()

plt.figure(figsize=(10,8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)

plt.title("Correlation Between Molecular Descriptors")

plt.show()

In [ ]:
##############################################################
# 5.10 Drug Diversity
##############################################################

print("=" * 60)
print("Drug Diversity")
print("=" * 60)

for col in columns:

    print(
        f"{col:<18}"
        f" Min = {drug_df[col].min():8.2f}"
        f" Max = {drug_df[col].max():8.2f}"
        f" Mean = {drug_df[col].mean():8.2f}"
    )

# Section 6: Protein Sequence Analysis

The Davis dataset represents each target as an amino acid sequence.

Unlike traditional bioinformatics pipelines, this project uses the pretrained ESM2 protein language model to learn protein representations directly from the raw sequence.

Therefore, instead of manually engineering biochemical descriptors, this section focuses on:

- Protein sequence statistics
- Sequence length analysis
- Amino acid composition
- Sequence quality assessment
- Preparation for ESM2 embeddings

These analyses help us understand the biological characteristics of the dataset while keeping the preprocessing aligned with modern protein foundation models.

In [ ]:
##############################################################
# 6.1 Imports
##############################################################

from collections import Counter

In [ ]:
##############################################################
# 6.2 Protein Sequence Length
##############################################################

protein_df["Sequence_Length"] = protein_df["Sequence"].str.len()

print("=" * 60)
print("Protein Sequence Statistics")
print("=" * 60)

display(protein_df.head())

print()

print(protein_df["Sequence_Length"].describe())

In [ ]:
##############################################################
# 6.3 Protein Length Distribution
##############################################################

plt.figure(figsize=(8,5))

plt.hist(
    protein_df["Sequence_Length"],
    bins=25,
    edgecolor="black"
)

plt.xlabel("Sequence Length")
plt.ylabel("Count")
plt.title("Protein Sequence Length Distribution")

plt.show()

In [ ]:
##############################################################
# 6.4 Amino Acid Composition
##############################################################

aa_counter = Counter()

for sequence in protein_df["Sequence"]:

    aa_counter.update(sequence)

aa_df = (
    pd.DataFrame(
        {
            "Residue": list(aa_counter.keys()),
            "Count": list(aa_counter.values())
        }
    )
    .sort_values("Count", ascending=False)
    .reset_index(drop=True)
)

display(aa_df)

In [ ]:
##############################################################
# 6.5 Amino Acid Frequency
##############################################################

plt.figure(figsize=(10,5))

plt.bar(
    aa_df["Residue"],
    aa_df["Count"]
)

plt.xlabel("Residue")
plt.ylabel("Frequency")
plt.title("Amino Acid Distribution")

plt.show()

In [ ]:
##############################################################
# 6.6 Sequence Quality Check
##############################################################

standard_amino_acids = set("ACDEFGHIKLMNPQRSTVWY")

all_residues = set()

for seq in protein_df["Sequence"]:

    all_residues.update(seq)

print("=" * 60)
print("Residues Present in Dataset")
print("=" * 60)

print(sorted(all_residues))

non_standard = sorted(
    all_residues - standard_amino_acids
)

print()

print("Non-standard Residues")

print(non_standard if non_standard else "None")

In [ ]:
##############################################################
# 6.7 Proteins Containing X
##############################################################

protein_df["Contains_X"] = protein_df["Sequence"].str.contains("X")

print("=" * 60)
print("Unknown Residues")
print("=" * 60)

print(
    protein_df["Contains_X"].value_counts()
)

display(
    protein_df[
        protein_df["Contains_X"]
    ].head()
)

In [ ]:
##############################################################
# 6.8 Length Categories
##############################################################

bins = [0, 250, 500, 750, 1000, 2000]

labels = [

    "0-250",

    "251-500",

    "501-750",

    "751-1000",

    "1000+"

]

protein_df["Length_Group"] = pd.cut(

    protein_df["Sequence_Length"],

    bins=bins,

    labels=labels

)

display(

    protein_df["Length_Group"].value_counts()

)

In [ ]:
##############################################################
# 6.9 Longest and Shortest Proteins
##############################################################

longest = protein_df.loc[
    protein_df["Sequence_Length"].idxmax()
]

shortest = protein_df.loc[
    protein_df["Sequence_Length"].idxmin()
]

print("=" * 60)

print("Longest Protein")

print("=" * 60)

display(longest)

print("=" * 60)

print("Shortest Protein")

print("=" * 60)

display(shortest)

In [ ]:
##############################################################
# 6.10 Protein Dataset Summary
##############################################################

print("=" * 60)
print("Protein Dataset Summary")
print("=" * 60)

print(f"Total Proteins : {len(protein_df):,}")

print(
    f"Average Length : "
    f"{protein_df['Sequence_Length'].mean():.2f}"
)

print(
    f"Maximum Length : "
    f"{protein_df['Sequence_Length'].max()}"
)

print(
    f"Minimum Length : "
    f"{protein_df['Sequence_Length'].min()}"
)

# Section 7: Molecular Graph Construction

Graph Neural Networks operate on graph-structured data.

Drug molecules are naturally represented as graphs:

- Atoms become graph nodes.
- Chemical bonds become graph edges.

Each atom is described by a set of node features, while each bond is described by edge features.

The resulting graph is stored as a PyTorch Geometric `Data` object that will later be used by the Graph Neural Network.

In [ ]:
##############################################################
# 7.1 Imports
##############################################################

from rdkit import Chem

import torch

from torch_geometric.data import Data

from tqdm.auto import tqdm

In [ ]:
##############################################################
# 7.2 Verify RDKit Molecules
##############################################################

print("="*60)
print("Verifying Molecules")
print("="*60)

invalid = drug_df["Molecule"].isnull().sum()

print(f"Invalid Molecules : {invalid}")

assert invalid == 0

print("All molecules are valid.")

In [ ]:
##############################################################
# 7.3 Atom Features
##############################################################

def atom_features(atom):

    return [

        atom.GetAtomicNum(),

        atom.GetDegree(),

        atom.GetFormalCharge(),

        atom.GetTotalNumHs(),

        int(atom.GetIsAromatic()),

        atom.GetHybridization().real,

        atom.GetImplicitValence(),

        atom.GetTotalValence(),

        atom.GetMass(),

    ]

In [ ]:
##############################################################
# 7.4 Bond Features
##############################################################

def bond_features(bond):

    return [

        bond.GetBondTypeAsDouble(),

        int(bond.GetIsConjugated()),

        int(bond.IsInRing()),

        int(bond.GetStereo())

    ]

In [ ]:
##############################################################
# 7.5 Convert Molecule to Graph
##############################################################

def molecule_to_graph(mol):

    node_features = []

    for atom in mol.GetAtoms():

        node_features.append(

            atom_features(atom)

        )

    x = torch.tensor(

        node_features,

        dtype=torch.float

    )

    edge_index = []

    edge_attr = []

    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()

        j = bond.GetEndAtomIdx()

        feature = bond_features(bond)

        edge_index.append([i, j])

        edge_index.append([j, i])

        edge_attr.append(feature)

        edge_attr.append(feature)

    edge_index = torch.tensor(

        edge_index,

        dtype=torch.long

    ).t().contiguous()

    edge_attr = torch.tensor(

        edge_attr,

        dtype=torch.float

    )

    return Data(

        x=x,

        edge_index=edge_index,

        edge_attr=edge_attr

    )

In [ ]:
##############################################################
# 7.6 Build Graph Dataset
##############################################################

graph_dataset = []

for mol in tqdm(drug_df["Molecule"]):

    graph_dataset.append(

        molecule_to_graph(mol)

    )

print()

print("Graphs Built :", len(graph_dataset))

In [ ]:
##############################################################
# 7.7 Example Graph
##############################################################

graph = graph_dataset[0]

print(graph)

print()

print("Node Features Shape")

print(graph.x.shape)

print()

print("Edge Index Shape")

print(graph.edge_index.shape)

print()

print("Edge Features Shape")

print(graph.edge_attr.shape)

In [ ]:
##############################################################
# 7.8 Graph Statistics
##############################################################

num_nodes = []

num_edges = []

for graph in graph_dataset:

    num_nodes.append(graph.num_nodes)

    num_edges.append(graph.num_edges)

print("="*60)
print("Graph Dataset Statistics")
print("="*60)

print(f"Average Nodes : {np.mean(num_nodes):.2f}")

print(f"Average Edges : {np.mean(num_edges):.2f}")

print(f"Maximum Nodes : {max(num_nodes)}")

print(f"Maximum Edges : {max(num_edges)}")

In [ ]:
##############################################################
# 7.9 Save Graph Dataset
##############################################################

GRAPH_DIR = PROCESSED_DIR / "graphs"

GRAPH_DIR.mkdir(
    parents=True,
    exist_ok=True
)

torch.save(

    graph_dataset,

    GRAPH_DIR / "drug_graphs.pt"

)

print("Graphs saved successfully.")

# Section 8: Advanced Molecular Graph Representation

Drug molecules are represented as graphs where:

- Nodes correspond to atoms.
- Edges correspond to chemical bonds.

Instead of using only numerical atom descriptors, we build a richer representation that captures:

- Atom identity
- Chemical environment
- Aromaticity
- Hybridization
- Chirality
- Ring membership
- Bond characteristics

This representation provides more chemically meaningful input to the Graph Neural Network.

In [ ]:
##############################################################
# 8.1 Imports
##############################################################
from rdkit import Chem

from rdkit.Chem.rdchem import HybridizationType
from rdkit.Chem.rdchem import BondType

from torch_geometric.data import Data

import torch
from tqdm.auto import tqdm

In [ ]:
##############################################################
# 8.2 Feature Vocabulary
##############################################################

ATOM_LIST = [
    "C",
    "N",
    "O",
    "S",
    "F",
    "P",
    "Cl",
    "Br",
    "I",
    "H",
    "Unknown"
]

HYBRIDIZATION_LIST = [

    HybridizationType.SP,

    HybridizationType.SP2,

    HybridizationType.SP3,

    HybridizationType.SP3D,

    HybridizationType.SP3D2

]

BOND_LIST = [

    BondType.SINGLE,

    BondType.DOUBLE,

    BondType.TRIPLE,

    BondType.AROMATIC

]

In [ ]:
##############################################################
# 8.3 Utility
##############################################################

def one_hot(value, choices):

    encoding = [0] * len(choices)

    if value in choices:

        encoding[choices.index(value)] = 1

    return encoding

In [ ]:
##############################################################
# 8.4 Atom Features
##############################################################

def atom_features(atom):

    symbol = atom.GetSymbol()

    if symbol not in ATOM_LIST:

        symbol = "Unknown"

    features = []

    features += one_hot(symbol, ATOM_LIST)

    features += one_hot(

        atom.GetHybridization(),

        HYBRIDIZATION_LIST

    )

    features += [

        atom.GetAtomicNum(),

        atom.GetDegree(),

        atom.GetFormalCharge(),

        atom.GetTotalNumHs(),

        atom.GetTotalValence(),

        int(atom.GetIsAromatic()),

        int(atom.IsInRing())

    ]

    return features

In [ ]:
##############################################################
# 8.5 Bond Features
##############################################################

def bond_features(bond):

    features = []

    features += one_hot(

        bond.GetBondType(),

        BOND_LIST

    )

    features += [

        int(bond.GetIsConjugated()),

        int(bond.IsInRing())

    ]

    return features

In [ ]:
##############################################################
# 8.6 Graph Builder
##############################################################

def build_graph(mol):

    x = []

    edge_index = []

    edge_attr = []

    for atom in mol.GetAtoms():

        x.append(

            atom_features(atom)

        )

    for bond in mol.GetBonds():

        i = bond.GetBeginAtomIdx()

        j = bond.GetEndAtomIdx()

        bf = bond_features(bond)

        edge_index.append([i, j])

        edge_index.append([j, i])

        edge_attr.append(bf)

        edge_attr.append(bf)

    x = torch.tensor(

        x,

        dtype=torch.float

    )

    edge_index = torch.tensor(

        edge_index,

        dtype=torch.long

    ).t().contiguous()

    edge_attr = torch.tensor(

        edge_attr,

        dtype=torch.float

    )

    return Data(

        x=x,

        edge_index=edge_index,

        edge_attr=edge_attr

    )

In [ ]:
##############################################################
# 8.7 Build Dataset
##############################################################

graph_dataset = []

for mol in tqdm(drug_df["Molecule"]):

    graph_dataset.append(

        build_graph(mol)

    )

print()

print("Graphs Created :", len(graph_dataset))

In [ ]:
##############################################################
# 8.8 Validation
##############################################################

for i, graph in enumerate(graph_dataset):

    assert graph.x.shape[0] == graph.num_nodes

    assert graph.edge_index.shape[0] == 2

    assert graph.edge_attr.shape[0] == graph.edge_index.shape[1]

print("All graphs validated successfully.")

In [ ]:
##############################################################
# 8.9 Save Graph Dataset
##############################################################

GRAPH_DIR = PROCESSED_DIR / "graphs"

GRAPH_DIR.mkdir(

    parents=True,

    exist_ok=True

)

torch.save(

    graph_dataset,

    GRAPH_DIR / "drug_graphs.pt"

)

print("Graph dataset saved.")

# Section 9: Protein Embedding Generation

The protein branch of our model uses a pretrained Protein Language Model (PLM).

Instead of manually engineering biochemical descriptors, we leverage Meta AI's ESM2 model, which has been pretrained on millions of protein sequences.

The workflow is:

1. Tokenize protein sequences.
2. Pass tokens through the frozen ESM2 encoder.
3. Apply mean pooling to obtain a fixed-length embedding.
4. Cache embeddings to disk.
5. Reuse cached embeddings during training.

Since the Davis dataset contains only 442 unique proteins, generating embeddings once is computationally efficient.

In [ ]:
##############################################################
# 9.1 Imports
##############################################################

from pathlib import Path

import torch

from transformers import AutoTokenizer
from transformers import EsmModel

from tqdm.auto import tqdm

In [ ]:
##############################################################
# 9.2 Configuration
##############################################################

MODEL_NAME = "facebook/esm2_t30_150M_UR50D"

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

MAX_LENGTH = 1024

BATCH_SIZE = 16

EMBEDDING_DIR = PROCESSED_DIR / "protein_embeddings"

EMBEDDING_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print(DEVICE)

In [ ]:
##############################################################
# 9.3 Load ESM2
##############################################################

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME
)

model = EsmModel.from_pretrained(
    MODEL_NAME
)

model.to(DEVICE)

model.eval()

print("ESM2 loaded successfully.")

In [ ]:
##############################################################
# 9.4 Prepare Protein Sequences
##############################################################

protein_sequences = protein_df["Sequence"].tolist()

protein_ids = protein_df["Protein_ID"].tolist()

print("Proteins :", len(protein_sequences))

In [ ]:
##############################################################
# 9.5 Mean Pooling
##############################################################

def mean_pool(last_hidden_state, attention_mask):

    mask = attention_mask.unsqueeze(-1)

    masked_embeddings = last_hidden_state * mask

    summed = masked_embeddings.sum(dim=1)

    counts = mask.sum(dim=1)

    return summed / counts
    

In [ ]:
##############################################################
# 9.6 Generate Embeddings
##############################################################

protein_embeddings = {}

with torch.no_grad():

    for protein_id, sequence in tqdm(
        zip(protein_ids, protein_sequences),
        total=len(protein_sequences)
    ):

        inputs = tokenizer(
            sequence,
            return_tensors="pt",
            truncation=True,
            max_length=MAX_LENGTH
        )

        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        outputs = model(**inputs)

        embedding = mean_pool(
            outputs.last_hidden_state,
            inputs["attention_mask"]
        )

        protein_embeddings[protein_id] = (
            embedding.squeeze(0)
            .cpu()
        )

In [ ]:
##############################################################
# 9.7 Save Embeddings
##############################################################

torch.save(

    protein_embeddings,

    EMBEDDING_DIR / "esm2_embeddings.pt"

)

print("Embeddings saved successfully.")

In [ ]:
##############################################################
# 9.8 Verification
##############################################################

sample_id = protein_ids[0]

embedding = protein_embeddings[sample_id]

print("=" * 60)
print("Embedding Verification")
print("=" * 60)

print("Protein ID :", sample_id)

print("Embedding Shape :", embedding.shape)

In [ ]:
##############################################################
# 9.9 Reload Test
##############################################################

loaded_embeddings = torch.load(
    EMBEDDING_DIR / "esm2_embeddings.pt"
)

print("Loaded :", len(loaded_embeddings))

# Section 10: Interaction Dataset

Our model predicts the interaction between a drug and a protein.

Each training example consists of:

- Drug molecular graph
- Protein embedding
- Binding affinity value

The Davis dataset contains:

- 68 drugs
- 442 proteins

forming approximately 30,000 drug–protein interaction pairs.

This section constructs a dataset where every sample combines these three components.

In [ ]:
##############################################################
# 10.1 Imports
##############################################################

import pickle

import numpy as np

import torch

from torch.utils.data import Dataset

In [ ]:
##############################################################
# 10.2 Load Processed Objects
##############################################################

graph_dataset = torch.load(
    GRAPH_DIR / "drug_graphs.pt",
    weights_only=False
)

protein_embeddings = torch.load(
    EMBEDDING_DIR / "esm2_embeddings.pt"
)

print("Drug Graphs :", len(graph_dataset))
print("Protein Embeddings :", len(protein_embeddings))

In [ ]:
##############################################################
# 10.3 Affinity Matrix
##############################################################

with open(
    DATASET_ROOT / "Y",
    "rb"
) as f:

    affinity_matrix = pickle.load(
        f,
        encoding="latin1"
    )

affinity_matrix = np.array(
    affinity_matrix
)

print(affinity_matrix.shape)

In [ ]:
##############################################################
# Convert Kd (nM) -> pKd
##############################################################

affinity_matrix = -np.log10(affinity_matrix / 1e9).astype(np.float32)

print("Affinity range after conversion:")
print(affinity_matrix.min(), affinity_matrix.max())

In [ ]:
##############################################################
# 10.4 Protein Mapping
##############################################################

protein_ids = list(
    protein_embeddings.keys()
)

protein_index = {

    idx: pid

    for idx, pid

    in enumerate(protein_ids)

}

In [ ]:
##############################################################
# 10.5 Dataset
##############################################################

class DTIDataset(Dataset):

    def __init__(

        self,

        drug_graphs,

        protein_embeddings,

        affinity_matrix

    ):

        self.samples = []

        for drug_idx in range(

            affinity_matrix.shape[0]

        ):

            for protein_idx in range(

                affinity_matrix.shape[1]

            ):

                affinity = affinity_matrix[

                    drug_idx,

                    protein_idx

                ]

                self.samples.append(

                    (

                        drug_idx,

                        protein_idx,

                        affinity

                    )

                )

        self.drug_graphs = drug_graphs

        self.protein_embeddings = protein_embeddings

    def __len__(self):

        return len(self.samples)

    def __getitem__(self, idx):

        drug_idx, protein_idx, affinity = self.samples[idx]

        graph = self.drug_graphs[drug_idx]

        protein = self.protein_embeddings[
            protein_index[protein_idx]
        ]

        return {

            "graph": graph,

            "protein": protein.float(),

            "label": torch.tensor(
                affinity,
                dtype=torch.float
            )

        }

In [ ]:
##############################################################
# 10.6 Build Dataset
##############################################################

dataset = DTIDataset(

    graph_dataset,

    protein_embeddings,

    affinity_matrix

)

print("Samples :", len(dataset))

In [ ]:
##############################################################
# 10.7 Example
##############################################################

sample = dataset[0]

print(sample["graph"])

print()

print(sample["protein"].shape)

print()

print(sample["label"])

In [ ]:
##############################################################
# 10.8 Statistics
##############################################################

labels = np.array(

    [

        sample[2]

        for sample in dataset.samples

    ]

)

print("="*60)

print("Interaction Dataset Statistics")

print("="*60)

print(f"Samples : {len(labels):,}")

print(f"Mean Affinity : {labels.mean():.3f}")

print(f"Std : {labels.std():.3f}")

print(f"Minimum : {labels.min():.3f}")

print(f"Maximum : {labels.max():.3f}")

# Section 11: Official Train/Test Split

The Davis benchmark provides predefined train and test folds.

Instead of randomly splitting the interaction matrix, we use these official folds to ensure our results are directly comparable with published Drug–Target Interaction models.

Each interaction pair is assigned to either:

- Training
- Validation
- Testing

using the benchmark protocol.

In [ ]:
##############################################################
# 11.1 Imports
##############################################################

from sklearn.model_selection import train_test_split

from torch.utils.data import Dataset

In [ ]:
train_file = DATASET_ROOT / "folds/train_fold_setting1.txt"

with open(train_file, "r") as f:
    print(f.read(200))

In [ ]:
##############################################################
# 11.2 Load Official Folds
##############################################################

import json

with open(DATASET_ROOT / "folds/train_fold_setting1.txt", "r") as f:
    train_folds = json.load(f)

with open(DATASET_ROOT / "folds/test_fold_setting1.txt", "r") as f:
    test_fold = json.load(f)

print("Training folds :", len(train_folds))
print("Test samples   :", len(test_fold))

In [ ]:
##############################################################
# 11.3 Validation Fold
##############################################################

VALIDATION_FOLD = 0

validation_indices = train_folds[VALIDATION_FOLD]

training_indices = []

for i, fold in enumerate(train_folds):

    if i != VALIDATION_FOLD:

        training_indices.extend(fold)

print(len(training_indices))

print(len(validation_indices))

In [ ]:
##############################################################
# 11.4 Index Conversion
##############################################################

num_drugs = affinity_matrix.shape[0]

num_proteins = affinity_matrix.shape[1]

def flat_to_pair(index):
    drug_idx = index // num_proteins
    protein_idx = index % num_proteins
    return drug_idx, protein_idx

# Debugging

In [ ]:
print("=" * 60)
print("Affinity before dataset creation")
print("=" * 60)

print(id(affinity_matrix))
print(affinity_matrix.min(), affinity_matrix.max())

In [ ]:
##############################################################
# 11.5 Dataset
##############################################################

class FoldDataset(Dataset):

    def __init__(

        self,

        indices,

        graphs,

        protein_embeddings,

        affinity_matrix

    ):

        self.indices = indices

        self.graphs = graphs

        self.embeddings = protein_embeddings

        self.affinity = affinity_matrix

        self.protein_keys = list(

            protein_embeddings.keys()

        )

    def __len__(self):

        return len(self.indices)

    def __getitem__(self, idx):

        flat_index = self.indices[idx]

        drug_idx, protein_idx = flat_to_pair(

            flat_index

        )

        graph = self.graphs[drug_idx]

        protein = self.embeddings[
            self.protein_keys[protein_idx]
        ]

        label = torch.tensor(

            self.affinity[
                drug_idx,
                protein_idx
            ],

            dtype=torch.float

        )

        return {

            "graph": graph,

            "protein": protein,

            "label": label

        }

In [ ]:
##############################################################
# 11.6 Create Datasets
##############################################################

train_dataset = FoldDataset(
    training_indices,
    graph_dataset,
    protein_embeddings,
    affinity_matrix
)

valid_dataset = FoldDataset(
    validation_indices,
    graph_dataset,
    protein_embeddings,
    affinity_matrix
)

test_dataset = FoldDataset(
    test_fold,
    graph_dataset,
    protein_embeddings,
    affinity_matrix
)

print()

print("Train :", len(train_dataset))

print("Valid :", len(valid_dataset))

print("Test  :", len(test_dataset))

In [ ]:
print("=" * 60)
print("Dataset after creation")
print("=" * 60)

print(id(train_dataset.affinity))
print(train_dataset.affinity.min(), train_dataset.affinity.max())

In [ ]:
##############################################################
# 11.7 Verification
##############################################################

sample = train_dataset[0]

print(sample["graph"])

print(sample["protein"].shape)

print(sample["label"])

In [ ]:
##############################################################
# 11.8 Statistics
##############################################################

print("="*60)

print("Fold Dataset")

print("="*60)

print(f"Train : {len(train_dataset):,}")

print(f"Valid : {len(valid_dataset):,}")

print(f"Test  : {len(test_dataset):,}")

Graph Dataset
        +
        
Protein Embeddings
        +
        
Affinity Labels
        │
        ▼
        
Custom Collate Function
        │
        ▼
        
PyTorch Geometric Batch
        │
        ▼

        
Training Loop

# Debugging


# Section 12: DataLoader Pipeline

The dataset contains three different data modalities:

- Molecular graphs
- Protein embeddings
- Binding affinity labels

Unlike conventional datasets, molecular graphs have different numbers of atoms and bonds.

PyTorch Geometric provides a specialized `DataLoader` that merges multiple graphs into one disconnected graph while preserving graph boundaries.

This section prepares efficient mini-batches for training.

In [ ]:
##############################################################
# 12.1 Imports
##############################################################

import torch

from torch_geometric.loader import DataLoader
from torch_geometric.data import Batch

In [ ]:
##############################################################
# 12.2 Custom Collate Function
##############################################################

def collate_fn(batch):

    graphs = [sample["graph"] for sample in batch]

    graphs = Batch.from_data_list(graphs)

    proteins = torch.stack(

        [

            sample["protein"]

            for sample in batch

        ]

    )

    labels = torch.stack(

        [

            sample["label"]

            for sample in batch

        ]

    )

    return {

        "graph": graphs,

        "protein": proteins,

        "label": labels

    }

In [ ]:
##############################################################
# 12.3 Configuration
##############################################################

BATCH_SIZE = 64

NUM_WORKERS = 2

PIN_MEMORY = True

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [ ]:
print(train_dataset.affinity.min(), train_dataset.affinity.max())

batch = next(iter(train_loader))
print(batch["label"][:10])

In [ ]:
##############################################################
# 12.5 Verify Batch
##############################################################

batch = next(iter(train_loader))

print("=" * 60)
print("Batch Verification")
print("=" * 60)

print(batch["graph"])
print()

print("Protein Shape")
print(batch["protein"].shape)

print()

print("Label Shape")
print(batch["label"].shape)

In [ ]:
##############################################################
# 12.6 Graph Batch Information
##############################################################

graph_batch = batch["graph"]

print("Number of graphs :", graph_batch.num_graphs)
print("Total nodes      :", graph_batch.num_nodes)
print("Total edges      :", graph_batch.num_edges)

In [ ]:
##############################################################
# 12.7 Device Test
##############################################################

DEVICE = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

graph = batch["graph"].to(DEVICE)

protein = batch["protein"].to(DEVICE)

label = batch["label"].to(DEVICE)

print("Device transfer successful!")

In [ ]:
##############################################################
# 12.8 DataLoader Benchmark
##############################################################

import time

start = time.time()

for i, batch in enumerate(train_loader):

    if i == 10:

        break

elapsed = time.time() - start

print(f"Time for 10 batches: {elapsed:.2f} seconds")

# Section 13: Drug Graph Encoder

This section implements the Graph Neural Network used to encode molecular graphs.

The encoder receives:

- Node features
- Edge indices
- Edge features

and produces a fixed-length embedding representing the entire drug molecule.

We use GINEConv because it naturally incorporates bond features during message passing, making it well suited for molecular graphs.

In [ ]:
##############################################################
# 13.1 Imports
##############################################################

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import (
    GINEConv,
    global_mean_pool,
    BatchNorm
)

In [ ]:
##############################################################
# 13.2 Configuration
##############################################################

NODE_DIM = graph_dataset[0].x.shape[1]

EDGE_DIM = graph_dataset[0].edge_attr.shape[1]

HIDDEN_DIM = 128

NUM_GNN_LAYERS = 3

DROPOUT = 0.2

In [ ]:
##############################################################
# 13.3 MLP Builder
##############################################################

def build_mlp(hidden_dim):
    return nn.Sequential(
        nn.Linear(hidden_dim, hidden_dim),
        nn.BatchNorm1d(hidden_dim),
        nn.ReLU(inplace=True),

        nn.Linear(hidden_dim, hidden_dim),
        nn.BatchNorm1d(hidden_dim),
        nn.ReLU(inplace=True),

        nn.Linear(hidden_dim, hidden_dim)
    )

In [ ]:
##############################################################
# 13.4 Drug Encoder
##############################################################

class DrugEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.node_projection = nn.Linear(
            NODE_DIM,
            HIDDEN_DIM
        )

        self.edge_projection = nn.Linear(
            EDGE_DIM,
            HIDDEN_DIM
        )

        self.convs = nn.ModuleList()

        self.norms = nn.ModuleList()

        for _ in range(NUM_GNN_LAYERS):

            self.convs.append(

                GINEConv(
                    nn=build_mlp(HIDDEN_DIM),
                    edge_dim=HIDDEN_DIM
                )

            )

            self.norms.append(

                BatchNorm(HIDDEN_DIM)

            )

        self.dropout = nn.Dropout(DROPOUT)

    def forward(
        self,
        graph
    ):

        x = self.node_projection(graph.x)

        edge_attr = self.edge_projection(
            graph.edge_attr
        )

        for conv, norm in zip(
            self.convs,
            self.norms
        ):

            x = conv(
                x,
                graph.edge_index,
                edge_attr
            )

            x = norm(x)

            x = F.relu(x)

            x = self.dropout(x)

        x = global_mean_pool(
            x,
            graph.batch
        )

        return x

In [ ]:
##############################################################
# 13.5 Verification
##############################################################

encoder = DrugEncoder().to(DEVICE)

batch = next(iter(train_loader))

graph = batch["graph"].to(DEVICE)

with torch.no_grad():

    embedding = encoder(graph)

print("=" * 60)

print("Drug Embedding Shape")

print("=" * 60)

print(embedding.shape)

In [ ]:
##############################################################
# 13.6 Model Size
##############################################################

params = sum(

    p.numel()

    for p in encoder.parameters()

)

trainable = sum(

    p.numel()

    for p in encoder.parameters()

    if p.requires_grad

)

print("=" * 60)

print("Drug Encoder")

print("=" * 60)

print(f"Parameters : {params:,}")

print(f"Trainable : {trainable:,}")

In [ ]:
##############################################################
# 13.7 Benchmark
##############################################################

import time

encoder.eval()

batch = next(iter(train_loader))

graph = batch["graph"].to(DEVICE)

start = time.time()

with torch.no_grad():

    for _ in range(100):

        encoder(graph)

elapsed = time.time() - start

print(f"Average forward time: {elapsed/100:.4f} sec")

# Section 14: Drug–Protein Fusion Network

The complete model combines information from both modalities:

- Drug molecular graph embeddings from the GINE encoder.
- Protein embeddings generated by the pretrained ESM2 model.

Before fusion, each modality is projected into a shared latent space.

The projected embeddings are concatenated and passed through a multilayer perceptron to predict the binding affinity.

In [ ]:
##############################################################
# 14.1 Imports
##############################################################

import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
##############################################################
# 14.2 Model Configuration
##############################################################

PROTEIN_DIM = 640

DRUG_DIM = 128

FUSION_DIM = 256

MLP_HIDDEN = 512

DROPOUT = 0.3

In [ ]:
##############################################################
# 14.3 Drug Target Interaction Model
##############################################################

class DTIModel(nn.Module):

    def __init__(self):

        super().__init__()

        self.drug_encoder = DrugEncoder()

        ########################################################
        # Drug Projection
        ########################################################

        self.drug_projection = nn.Sequential(

            nn.Linear(DRUG_DIM, FUSION_DIM),

            nn.BatchNorm1d(FUSION_DIM),

            nn.SiLU(),

            nn.Dropout(DROPOUT)

        )

        ########################################################
        # Protein Projection
        ########################################################

        self.protein_projection = nn.Sequential(

            nn.Linear(PROTEIN_DIM, FUSION_DIM),

            nn.BatchNorm1d(FUSION_DIM),

            nn.SiLU(),

            nn.Dropout(DROPOUT)

        )

        ########################################################
        # Fusion Network
        ########################################################

        self.fusion = nn.Sequential(

            nn.Linear(FUSION_DIM * 4, MLP_HIDDEN),

            nn.BatchNorm1d(MLP_HIDDEN),

            nn.SiLU(),

            nn.Dropout(DROPOUT),

            nn.Linear(MLP_HIDDEN, 512),

            nn.BatchNorm1d(512),

            nn.SiLU(),

            nn.Dropout(DROPOUT),

            nn.Linear(512, 256),

            nn.BatchNorm1d(256),

            nn.SiLU(),

            nn.Dropout(DROPOUT),

            nn.Linear(256, 1)

        )

    def forward(
        self,
        graph,
        protein
    ):

        ########################################################
        # Drug Encoder
        ########################################################

        drug_embedding = self.drug_encoder(graph)

        drug_embedding = self.drug_projection(
            drug_embedding
        )

        ########################################################
        # Protein Encoder
        ########################################################

        protein_embedding = self.protein_projection(
            protein
        )

        ########################################################
        # Pairwise Interaction Features
        ########################################################

        interaction = drug_embedding * protein_embedding

        difference = torch.abs(
            drug_embedding - protein_embedding
        )

        ########################################################
        # Fusion
        ########################################################

        fusion = torch.cat(
            [
                drug_embedding,
                protein_embedding,
                interaction,
                difference
            ],
            dim=1
        )

        ########################################################
        # Prediction
        ########################################################

        output = self.fusion(fusion)

        return output.squeeze(1)

In [ ]:
batch = next(iter(train_loader))

graph = batch["graph"].to(device)
protein = batch["protein"].to(device)

model = DTIModel().to(device)

with torch.no_grad():

    drug = model.drug_projection(model.drug_encoder(graph))

    prot = model.protein_projection(protein)

    interaction = drug * prot

    difference = torch.abs(drug - prot)

    fusion = torch.cat(
        [drug, prot, interaction, difference],
        dim=1
    )

print("Drug       :", drug.shape)
print("Protein    :", prot.shape)
print("Interaction:", interaction.shape)
print("Difference :", difference.shape)
print("Fusion     :", fusion.shape)
print("Expected   :", model.fusion[0].in_features)

In [ ]:
batch = next(iter(train_loader))

graph = batch["graph"].to(device)
protein = batch["protein"].to(device)

with torch.no_grad():
    pred = model(graph, protein)

In [ ]:
##############################################################
# 14.4 Initialize Model
##############################################################

model = DTIModel().to(DEVICE)

print(model)

In [ ]:
##############################################################
# 14.5 Forward Test
##############################################################

batch = next(iter(train_loader))

graph = batch["graph"].to(DEVICE)

protein = batch["protein"].to(DEVICE)

with torch.no_grad():

    prediction = model(

        graph,

        protein

    )

print("=" * 60)

print("Prediction Shape")

print("=" * 60)

print(prediction.shape)

In [ ]:
##############################################################
# 14.6 Model Parameters
##############################################################

total = sum(

    p.numel()

    for p in model.parameters()

)

trainable = sum(

    p.numel()

    for p in model.parameters()

    if p.requires_grad

)

print("=" * 60)

print("Model Statistics")

print("=" * 60)

print(f"Total Parameters      : {total:,}")

print(f"Trainable Parameters  : {trainable:,}")

In [ ]:
##############################################################
# 14.7 Model Summary
##############################################################

print("=" * 60)

print("Input")

print("=" * 60)

print("Drug Graph")

print("Protein Embedding : (640,)")

print()

print("=" * 60)

print("Output")

print("=" * 60)

print("Binding Affinity")

# Section 15: Training Utilities

This section prepares all components required for model training.

It includes:

- Loss function
- Optimizer
- Learning rate scheduler
- Metric tracking
- Checkpoint saving
- Automatic resume capability

These utilities make the training process reproducible and robust against interruptions.

In [ ]:
##############################################################
# 15.1 Imports
##############################################################

from pathlib import Path
import json

import torch
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau

In [ ]:
##############################################################
# 15.2 Training Configuration
##############################################################

LEARNING_RATE = 1e-3

WEIGHT_DECAY = 1e-4

NUM_EPOCHS = 50

CHECKPOINT_DIR = Path("checkpoints")

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

In [ ]:
##############################################################
# 15.3 Loss
##############################################################

criterion = nn.MSELoss()

In [ ]:
##############################################################
# 15.4 Optimizer
##############################################################

optimizer = AdamW(

    model.parameters(),

    lr=LEARNING_RATE,

    weight_decay=WEIGHT_DECAY

)

In [ ]:
##############################################################
# 15.5 Scheduler
##############################################################

scheduler = ReduceLROnPlateau(

    optimizer,

    mode="min",

    factor=0.5,

    patience=3,

    min_lr=1e-6
)

In [ ]:
##############################################################
# 15.6 Metric History
##############################################################

history = {

    "train_loss": [],

    "valid_loss": [],

    "learning_rate": []

}

In [ ]:
##############################################################
# 15.7 Save Checkpoint
##############################################################

def save_checkpoint(

    epoch,

    model,

    optimizer,

    scheduler,

    history,

    best_loss,

    filename

):

    checkpoint = {

        "epoch": epoch,

        "model_state_dict": model.state_dict(),

        "optimizer_state_dict": optimizer.state_dict(),

        "scheduler_state_dict": scheduler.state_dict(),

        "history": history,

        "best_loss": best_loss

    }

    torch.save(

        checkpoint,

        CHECKPOINT_DIR / filename

    )

In [ ]:
##############################################################
# 15.8 Load Checkpoint
##############################################################

def load_checkpoint(

    filename,

    model,

    optimizer,

    scheduler

):

    checkpoint = torch.load(

        CHECKPOINT_DIR / filename,

        map_location=DEVICE
    )

    model.load_state_dict(

        checkpoint["model_state_dict"]

    )

    optimizer.load_state_dict(

        checkpoint["optimizer_state_dict"]

    )

    scheduler.load_state_dict(

        checkpoint["scheduler_state_dict"]

    )

    return (

        checkpoint["epoch"],

        checkpoint["history"],

        checkpoint["best_loss"]

    )

In [ ]:
##############################################################
# 15.9 Save Best Model
##############################################################

def save_best_model(

    model

):

    torch.save(

        model.state_dict(),

        CHECKPOINT_DIR / "best_model.pt"

    )

In [ ]:
##############################################################
# 15.10 Save History
##############################################################

def save_history(history):

    with open(

        CHECKPOINT_DIR / "history.json",

        "w"

    ) as f:

        json.dump(

            history,

            f,

            indent=4
        )

In [ ]:
##############################################################
# 15.11 Resume
##############################################################

start_epoch = 0
best_loss = float("inf")
history = {
    "train_loss": [],
    "val_loss": [],
    "train_rmse": [],
    "val_rmse": [],
    "learning_rate": []
}

print("Training from scratch.")

In [ ]:
##############################################################
# 15.12 Verification
##############################################################

print("="*60)

print("Training Utilities Ready")

print("="*60)

print("Loss :", criterion)

print()

print("Optimizer")

print(optimizer)

print()

print("Scheduler")

print(scheduler)

print()

print("Checkpoint Directory")

print(CHECKPOINT_DIR)

# Section 16: Model Training

This section trains the Drug–Target Interaction model.

The training pipeline consists of:

- Mixed precision training (AMP)
- Gradient clipping
- Learning rate scheduling
- Checkpoint saving
- Validation after every epoch
- Automatic resume
- Early stopping

The model is optimized using Mean Squared Error (MSE) loss.

In [ ]:
##############################################################
# 16.1 Imports
##############################################################

from tqdm.auto import tqdm

import torch

from torch.amp import autocast
from torch.amp import GradScaler

from sklearn.metrics import mean_squared_error

In [ ]:
##############################################################
# 16.2 AMP
##############################################################

scaler = GradScaler(
    device="cuda",
    enabled=torch.cuda.is_available()
)

In [ ]:
##############################################################
# 16.3 Training Function
##############################################################

def train_one_epoch(
    model,
    loader,
    optimizer,
    criterion,
    scaler,
    device
):

    model.train()

    running_loss = 0.0

    progress = tqdm(
        loader,
        leave=False
    )

    for batch in progress:

        graph = batch["graph"].to(device)

        protein = batch["protein"].to(device)

        label = batch["label"].to(device)

        optimizer.zero_grad(set_to_none=True)

        with autocast(
            device_type=device.type,
            enabled=torch.cuda.is_available()
        ):

            prediction = model(
                graph,
                protein
            )
            

            loss = criterion(
                prediction,
                label
            )

        scaler.scale(loss).backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            1.0
        )

        scaler.step(optimizer)

        scaler.update()

        running_loss += loss.item()

        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return running_loss / len(loader)

In [ ]:
##############################################################
# 16.4 Validation
##############################################################

def validate_one_epoch(
    model,
    loader,
    criterion,
    device
):

    model.eval()

    running_loss = 0

    predictions = []

    targets = []

    with torch.no_grad():

        for batch in tqdm(
            loader,
            leave=False
        ):

            graph = batch["graph"].to(device)

            protein = batch["protein"].to(device)

            label = batch["label"].to(device)

            prediction = model(
                graph,
                protein
            )

            loss = criterion(
                prediction,
                label
            )

            running_loss += loss.item()

            predictions.extend(
                prediction.cpu().numpy()
            )

            targets.extend(
                label.cpu().numpy()
            )

    mse = mean_squared_error(
        targets,
        predictions
    )

    return (

        running_loss / len(loader),

        mse

    )

In [ ]:
##############################################################
# 16.5 Early Stopping
##############################################################

class EarlyStopping:

    def __init__(
        self,
        patience=10
    ):

        self.patience = patience

        self.best = float("inf")

        self.counter = 0

    def step(
        self,
        loss
    ):

        if loss < self.best:

            self.best = loss

            self.counter = 0

            return False

        self.counter += 1

        return self.counter >= self.patience

In [ ]:
##############################################################
# 16.6 Initialize
##############################################################

early_stopping = EarlyStopping(
    patience=7
)

In [ ]:
##############################################################
# 16.7 Main Training Loop
##############################################################
for epoch in range(
    start_epoch,
    NUM_EPOCHS
):

    print()

    print("="*70)

    print(
        f"Epoch {epoch+1}/{NUM_EPOCHS}"
    )

    print("="*70)

    train_loss = train_one_epoch(

        model,

        train_loader,

        optimizer,

        criterion,

        scaler,

        DEVICE

    )

    valid_loss, mse = validate_one_epoch(

        model,

        valid_loader,

        criterion,

        DEVICE

    )

    scheduler.step(
        valid_loss
    )

    history["train_loss"].append(
        train_loss
    )

    history["val_loss"].append(
        valid_loss
    )

    history["learning_rate"].append(

        optimizer.param_groups[0]["lr"]

    )

    print(
        f"Train Loss : {train_loss:.4f}"
    )

    print(
        f"Valid Loss : {valid_loss:.4f}"
    )

    print(
        f"MSE : {mse:.4f}"
    )

    if valid_loss < best_loss:

        best_loss = valid_loss

        save_best_model(model)

    save_checkpoint(

        epoch,

        model,

        optimizer,

        scheduler,

        history,

        best_loss,

        "latest.pt"

    )

    save_history(history)

    if early_stopping.step(
        valid_loss
    ):

        print()

        print("Early stopping triggered.")

        break

In [ ]:
print(history)

In [ ]:
print("start_epoch =", start_epoch)
print("NUM_EPOCHS  =", NUM_EPOCHS)

In [ ]:
batch = next(iter(train_loader))

print(batch.keys())
print(batch["label"][:10])

print("Min label:", batch["label"].min().item())
print("Max label:", batch["label"].max().item())

In [ ]:
import numpy as np

labels = []

for batch in train_loader:
    labels.extend(batch["label"].cpu().numpy())

labels = np.array(labels)

print("Unique largest values:")
print(np.unique(labels)[-20:])

print("Number of 10000 labels:", np.sum(labels == 10000))
print("Total labels:", len(labels))

# Section 17: Model Evaluation

This section evaluates the trained Drug–Target Interaction model on the official Davis test set.

We compute several standard regression metrics used in the literature:

- Mean Squared Error (MSE)
- Root Mean Squared Error (RMSE)
- Mean Absolute Error (MAE)
- Pearson Correlation
- R² Score
- Concordance Index (CI)

We also generate diagnostic plots to visualize model performance.

In [ ]:
##############################################################
# 17.1 Imports
##############################################################

import numpy as np

import matplotlib.pyplot as plt

from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    r2_score
)

from scipy.stats import pearsonr

In [ ]:
##############################################################
# 17.2 Prediction Function
##############################################################

def predict_dataset(
    model,
    loader,
    device
):

    model.eval()

    predictions = []

    targets = []

    with torch.no_grad():

        for batch in tqdm(loader):

            graph = batch["graph"].to(device)

            protein = batch["protein"].to(device)

            label = batch["label"].to(device)

            prediction = model(
                graph,
                protein
            )

            predictions.extend(
                prediction.cpu().numpy()
            )

            targets.extend(
                label.cpu().numpy()
            )

    predictions = np.array(predictions)

    targets = np.array(targets)

    return predictions, targets

In [ ]:
##############################################################
# 17.3 Concordance Index
##############################################################

def concordance_index(
    y_true,
    y_pred
):

    concordant = 0

    permissible = 0

    n = len(y_true)

    for i in range(n):

        for j in range(i + 1, n):

            if y_true[i] == y_true[j]:
                continue

            permissible += 1

            if (
                (y_pred[i] > y_pred[j])
                ==
                (y_true[i] > y_true[j])
            ):

                concordant += 1

            elif y_pred[i] == y_pred[j]:

                concordant += 0.5

    return concordant / permissible

In [ ]:
##############################################################
# 17.4 Test Prediction
##############################################################

predictions, targets = predict_dataset(

    model,

    test_loader,

    DEVICE

)

print("Prediction completed.")

In [ ]:
##############################################################
# 17.5 Metrics
##############################################################

mse = mean_squared_error(
    targets,
    predictions
)

rmse = np.sqrt(mse)

mae = mean_absolute_error(
    targets,
    predictions
)

r2 = r2_score(
    targets,
    predictions
)

pearson = pearsonr(
    targets,
    predictions
)[0]

ci = concordance_index(
    targets,
    predictions
)

print("=" * 60)
print("Test Metrics")
print("=" * 60)

print(f"MSE      : {mse:.4f}")
print(f"RMSE     : {rmse:.4f}")
print(f"MAE      : {mae:.4f}")
print(f"R²       : {r2:.4f}")
print(f"Pearson  : {pearson:.4f}")
print(f"CI       : {ci:.4f}")

In [ ]:
##############################################################
# 17.6 Scatter Plot
##############################################################

plt.figure(figsize=(7,7))

plt.scatter(
    targets,
    predictions,
    alpha=0.5
)

low = min(targets.min(), predictions.min())
high = max(targets.max(), predictions.max())

plt.plot(
    [low, high],
    [low, high],
    "r--",
    linewidth=2
)

plt.xlabel("Actual Affinity")
plt.ylabel("Predicted Affinity")
plt.title("Predicted vs Actual")

plt.show()

In [ ]:
##############################################################
# 17.7 Residuals
##############################################################

residuals = predictions - targets

plt.figure(figsize=(8,5))

plt.hist(
    residuals,
    bins=40,
    edgecolor="black"
)

plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Residual Distribution")

plt.show()

In [ ]:
##############################################################
# 17.8 Save Results
##############################################################

results = {

    "MSE": float(mse),

    "RMSE": float(rmse),

    "MAE": float(mae),

    "R2": float(r2),

    "Pearson": float(pearson),

    "CI": float(ci)

}

with open(

    CHECKPOINT_DIR / "test_results.json",

    "w"

) as f:

    json.dump(

        results,

        f,

        indent=4

    )

print("Evaluation results saved.")

# Section 18: Inference Pipeline

This section demonstrates how to use the trained Drug–Target Interaction model to predict the binding affinity of a completely new drug–protein pair.

The inference pipeline performs:

1. SMILES → Molecular Graph
2. Protein Sequence → ESM2 Embedding
3. Graph + Embedding → DTI Model
4. Predicted Binding Affinity

In [ ]:
##############################################################
# 18.1 Load Best Model
##############################################################

model = DTIModel().to(DEVICE)

checkpoint = torch.load(

    CHECKPOINT_DIR / "best_model.pt",

    map_location=DEVICE

)

# Support both full checkpoint and state_dict
if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
    model.load_state_dict(checkpoint["model_state_dict"])
else:
    model.load_state_dict(checkpoint)

model.eval()

print("Best model loaded.")

In [ ]:
##############################################################
# 18.2 Build Drug Graph
##############################################################

def smiles_to_graph(smiles):

    mol = Chem.MolFromSmiles(smiles)

    if mol is None:

        raise ValueError("Invalid SMILES.")

    graph = build_graph(mol)

    graph.batch = torch.zeros(

        graph.num_nodes,

        dtype=torch.long

    )

    return graph

In [ ]:
##############################################################
# 18.3 Protein Embedding
##############################################################

@torch.no_grad()

def embed_protein(sequence):

    inputs = tokenizer(

        sequence,

        return_tensors="pt",

        truncation=True,

        max_length=1024

    )

    inputs = {

        k: v.to(DEVICE)

        for k, v in inputs.items()

    }

    outputs = model_esm(**inputs)

    embedding = mean_pool(

        outputs.last_hidden_state,

        inputs["attention_mask"]

    )

    return embedding

In [ ]:
##############################################################
# 18.4 Prediction
##############################################################

@torch.no_grad()

def predict_affinity(

    smiles,

    sequence

):

    graph = smiles_to_graph(

        smiles

    ).to(DEVICE)

    protein = embed_protein(

        sequence

    )

    prediction = model(

        graph,

        protein

    )

    return prediction.item()

In [ ]:
##############################################################
# 18.5 Example
##############################################################

example_smiles = "CCO"

example_sequence = (
    "MTEITAAMVKELRESTGAGMMDCKNALSETQHE..."
)

prediction = predict_affinity(

    example_smiles,

    example_sequence

)

print("="*60)

print("Predicted Binding Affinity")

print("="*60)

print(prediction)

In [ ]:
##############################################################
# 18.6 Multiple Predictions
##############################################################

pairs = [

    {

        "smiles":"CCO",

        "sequence":"..."

    },

    {

        "smiles":"CCN",

        "sequence":"..."

    }

]

results = []

for pair in pairs:

    affinity = predict_affinity(

        pair["smiles"],

        pair["sequence"]

    )

    results.append(

        affinity

    )

print(results)

In [ ]:
##############################################################
# 18.7 Save Results
##############################################################

prediction_df = pd.DataFrame(

    {

        "SMILES":[

            p["smiles"]

            for p in pairs

        ],

        "Prediction":results

    }

)

prediction_df.to_csv(

    CHECKPOINT_DIR / "predictions.csv",

    index=False

)

prediction_df.head()

# Section 19: Model Explainability

Deep learning models often behave as black boxes.

To improve interpretability, we explain the predictions made by the molecular graph encoder using PyTorch Geometric's `Explainer`.

The explainer identifies:

- Important atoms
- Important chemical bonds
- Important molecular substructures

This helps us understand which parts of the molecule contribute most to the predicted binding affinity.

In [ ]:
##############################################################
# 19.1 Imports
##############################################################

from torch_geometric.explain import (
    Explainer,
    GNNExplainer,
    ModelConfig
)

import matplotlib.pyplot as plt

from rdkit.Chem import Draw

In [ ]:
##############################################################
# 19.2 Wrapper
##############################################################

from torch_geometric.nn import (
    GINEConv,
    BatchNorm,
    global_mean_pool,
    global_max_pool
)

class DrugEncoder(nn.Module):

    def __init__(self):

        super().__init__()

        self.node_projection = nn.Linear(NODE_DIM, HIDDEN_DIM)
        self.edge_projection = nn.Linear(EDGE_DIM, HIDDEN_DIM)

        self.convs = nn.ModuleList()
        self.norms = nn.ModuleList()

        for _ in range(NUM_GNN_LAYERS):

            self.convs.append(
                GINEConv(
                    nn=build_mlp(HIDDEN_DIM),
                    edge_dim=HIDDEN_DIM
                )
            )

            self.norms.append(
                BatchNorm(HIDDEN_DIM)
            )

        self.dropout = nn.Dropout(DROPOUT)

        self.final_projection = nn.Sequential(
            nn.Linear(HIDDEN_DIM * 2, HIDDEN_DIM),
            nn.BatchNorm1d(HIDDEN_DIM),
            nn.SiLU(),
            nn.Dropout(DROPOUT)
        )

    def forward(self, graph):

        x = self.node_projection(graph.x)
        edge_attr = self.edge_projection(graph.edge_attr)

        for conv, norm in zip(self.convs, self.norms):

            residual = x

            x = conv(
                x,
                graph.edge_index,
                edge_attr
            )

            x = norm(x)

            x = F.silu(x)

            x = x + residual

            x = self.dropout(x)

        mean_pool = global_mean_pool(
            x,
            graph.batch
        )

        max_pool = global_max_pool(
            x,
            graph.batch
        )

        x = torch.cat(
            [mean_pool, max_pool],
            dim=1
        )

        x = self.final_projection(x)

        return x

In [ ]:
##############################################################
# 19.3 Explainer
##############################################################

drug_encoder = model.drug_encoder.eval()

wrapper = DrugEncoderWrapper(
    drug_encoder
)

explainer = Explainer(

    model=wrapper,

    algorithm=GNNExplainer(
        epochs=200
    ),

    explanation_type="model",

    node_mask_type="attributes",

    edge_mask_type="object",

    model_config=ModelConfig(

        mode="regression",

        task_level="graph",

        return_type="raw"

    )

)

In [ ]:
##############################################################
# 19.4 Example Graph
##############################################################

graph = graph_dataset[0].to(DEVICE)

In [ ]:
##############################################################
# 19.5 Explain
##############################################################

explanation = explainer(

    x=graph.x,

    edge_index=graph.edge_index,

    edge_attr=graph.edge_attr,

    batch=torch.zeros(
        graph.num_nodes,
        dtype=torch.long,
        device=DEVICE
    )

)

In [ ]:
##############################################################
# 19.6 Feature Importance
##############################################################

print("Node Importance")

print(explanation.node_mask.shape)

print()

print("Edge Importance")

print(explanation.edge_mask.shape)

In [ ]:
##############################################################
# 19.7 Top Bonds
##############################################################

edge_scores = explanation.edge_mask.cpu().numpy()

top_edges = edge_scores.argsort()[-10:]

print("Most Important Bonds")

print(top_edges)

In [ ]:
##############################################################
# 19.8 Molecule
##############################################################

mol = drug_df.loc[0, "Molecule"]

image = Draw.MolToImage(
    mol,
    size=(500,500)
)

plt.imshow(image)

plt.axis("off")

plt.show()

In [ ]:
##############################################################
# 19.9 Save
##############################################################

torch.save(

    explanation,

    CHECKPOINT_DIR / "sample_explanation.pt"

)

print("Explanation saved.")